# Res model QC (standard template)

Run by `opmagent` for every res model version; the HTML export is the version's QC report and the approval is tied to its sha256. **Read only:** the notebook loads the build payload, the res tables from the database (SWOF, SGOF, rock, contacts, aquifers) and the geo version's static view made with this version's regions (its INIT: pore volume, depth and region arrays per cell, as OPM Flow sees them). It computes checks and plots and writes `qc_summary.json`. Don't edit this template for one model: start an investigation notebook instead.

In [ ]:
payload_path = "payload.json"   # written by the res pipeline
db_path = ""                    # SQLite database (opened read-only)
version_id = 0                  # res_model_versions.id
summary_path = "qc_summary.json"

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'   # self-contained HTML export
from IPython.display import display
from opmagent.notebooks import res_qc_lib as qc

data = qc.load(payload_path, db_path, version_id)
findings = qc.run_all_checks(data)
qc.header(data)

## Summary

Every QC finding, worst first. Fail and warn findings don't block the build; they are for the reviewer.

In [ ]:
qc.show_findings(findings)

## Assumptions and validator warnings

In [ ]:
display(qc.show_list(data.get('assumptions', []), title='Assumptions'))
qc.show_list(title='Validator warnings', items=[w['message'] + (f" ({w['where']})" if w.get('where') else '') for w in data.get('validator_warnings', [])])

## 1. Relative permeability and capillary pressure

End points per SATNUM: connate water, residual oil to water, krw there, the mobile oil window, critical gas and residual oil to gas. kro at connate water must be the same in SWOF and SGOF (Flow's three-phase model starts from it), SGOF must end within 1 - Swc, and Pcow should not rise with Sw.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'scal']))
display(qc.end_points(data).round(4))
display(qc.fig_kr(data))
f = qc.fig_pc(data)
if f is not None:
    display(f)

## 2. Regions

The region arrays as Flow sees them (from the static view): every SATNUM and EQLNUM value has its table, every table is used, and each FIPNUM region holds pore volume.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'regions']))
for kw in ('SATNUM', 'EQLNUM', 'FIPNUM'):
    v = qc.region_volumes(data, kw)
    if len(v):
        display(v.round(1))
        f = qc.fig_regions(data, kw)
        if f is not None:
            display(f)

## 3. Contacts and initial pressure

Per equilibration region: the datum within the region's cell depths, the GOC above the OWC, and where each contact lies against the cells (inside, above or below the region).

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] in ('contacts', 'rock')]))
f = qc.fig_contacts(data)
if f is not None:
    display(f)

## 4. Analytical aquifers

Each aquifer's connected cells must be active (Flow skips inactive ones), and its size is set against the reservoir's pore volume: a Fetkovich aquifer smaller than the reservoir gives little support.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'aquifers']))
t = qc.aquifer_table(data)
if len(t):
    display(t.round(3))
    f = qc.fig_aquifers(data)
    if f is not None:
        display(f)

## Result

In [ ]:
summary = qc.write_summary(summary_path, findings, data)
print(f"Overall: {summary['overall'].upper()}  " + ", ".join(f'{k} {v}' for k, v in summary['counts'].items()))